# Panel-SINDy for corporate financial distress — Kaggle run

Structure discovery on the Indonesian listed-firm panel (324 firms x 11 years), following
`RESEARCH_DESIGN.md`. One notebook, top to bottom, unattended.

| Part | Covers | Design ref |
|---|---|---|
| **1** | State coordinates, and the boundary fix | §2 |
| **2** | `panel_stlsq` — discrete-time sparse regression with firm effects and group sparsity | §4.1 |
| **3** | **Synthetic recovery test** — can the estimator recover a known map at this N, T, noise? | §10 |
| **4** | **E1** — library / threshold sweep. The gate experiment; it decides the narrative | §5 E1, §7 |
| **5** | **E2** — baseline ladder with paired bootstrap CIs | §5 E2 |
| **6** | **E3** — cohort contrast with permutation inference and E-SINDy inclusion probabilities | §5 E3 |
| **7** | **E5** — first-passage validation against the "rank by current IEQ" bar | §5 E5 |

Part 3 is a gate in the strict sense: **if the estimator cannot recover a known sparse map under
exactly this design, nothing in Parts 4–7 means anything.** It runs first and prints a verdict.

E4 (unsupervised regime discovery), E6 (SINDyc / year effects), E7 (the synthetic Δt/τ
admissibility study) and E8–E9 (governance strata, phase portraits) are **not** in this notebook.
E7 in particular is a standalone simulation study with no dependence on this data and belongs in
its own notebook.

---

## Kaggle setup

**1. Attach the data.** The notebook needs `panel_sindy_ready.csv` (in
`notebooks/financialdist/`, next to this file). Either:

* upload it via *Datasets → New Dataset* and *Add Data* it to the notebook — it will be found
  anywhere under `/kaggle/input`; or
* set the `FINDIST_DATA` environment variable to the directory holding it.

No internet access and no GPU are needed. Everything is numpy + pandas; `scikit-learn` is used
only for the random-forest rung of E2 and is skipped cleanly if unavailable.

**2. Set `QUICK`.** `QUICK = True` shrinks every grid so the whole notebook runs in ~2 minutes —
use it once to check the environment. `QUICK = False` is the real run: roughly **15–30 minutes**
on a Kaggle CPU, dominated by the E2 random forest and the 5,000 E3 permutations.

**3. Read `results_financialdist.json`** in `/kaggle/working` afterwards. Every number the
write-up needs is in there, alongside the figures.

---

## One thing that changed relative to the shipped data

`panel_sindy_ready.csv` ships `*_z` columns that are standardised to zero mean. For eleven of the
twelve states that is exactly right. For `IEQ` it is not: the distress boundary is `IEQ = 0`, and
subtracting the mean moves that boundary off the origin. Part 1 rebuilds the state from the `*_t`
(asinh) columns with `IEQ` scaled but **not** centred, so `IEQ_state = 0` is the boundary, and
prints the diagnostic that shows why this matters. See `DATA_REPORT.md` §3.

## 0. Configuration

`QUICK = True` shrinks every expensive knob so the whole notebook runs in a few minutes. Use it to
validate the environment, then set it back to `False`.

In [ ]:
import os, json, time, math
from pathlib import Path

# Printed first and stored in the results JSON, so a log always says which version ran.
# Kaggle re-runs the notebook saved in its editor, not the file on disk -- if this line does
# not match the repo, re-import before reading anything below.
NOTEBOOK_VERSION = "2026-08-09  panel-SINDy v1: boundary-centred IEQ, FE + group sparsity"

# ----------------------------------------------------------------- knobs
QUICK = True           # True -> ~2 min smoke test; False -> the real run (~15-30 min CPU)

RUN_SYNTH = True       # Part 3, the estimator gate. Leave on: it is cheap and it is the gate.
RUN_E1    = True       # Part 4, library/threshold sweep -- decides the narrative (design §7)
RUN_E2    = True       # Part 5, baseline ladder
RUN_E3    = True       # Part 6, cohort contrast + permutation inference
RUN_E5    = True       # Part 7, first-passage validation

SEED = 0

# Held-out firms have no estimated firm effect. We give them a burn-in window and read
# alpha_i off it, then start every rollout at or after the end of that window, so the
# firm effect never sees the target. T_BURN = 6 is 2013-2018, matching E5's split.
T_BURN   = 6
H_SELECT = 3           # horizon used for model selection (design §4.2: NOT one-step R2)
HORIZONS = (1, 3, 5)
CLIP     = 8.0         # state box for rollouts, in z units; the clipping rate is reported

N_OUTER, N_INNER = 5, 4
N_SEEDS_E2 = 20
N_PERM_E3  = 5000      # pre-registered in design §5 E3
N_BOOT_E3  = 500       # E-SINDy bootstraps over firms, per cohort
N_PATHS_E5 = 10000

# Hyperparameter grid. THRESHOLD_GRID is (log10 lo, log10 hi, count).
THRESHOLD_GRID = (-3.0, 0.0, 12)
ALPHA_GRID     = (1e-4, 1e-3, 1e-2)
LIBRARIES      = ("linear", "lin+sq", "deg2r", "deg2")

if QUICK:
    N_OUTER, N_INNER = 3, 2
    N_SEEDS_E2 = 3
    N_PERM_E3, N_BOOT_E3 = 200, 100
    N_PATHS_E5 = 1000
    THRESHOLD_GRID = (-3.0, 0.0, 7)
    ALPHA_GRID     = (1e-3, 1e-2)
    LIBRARIES      = ("linear", "deg2")

print(f"NOTEBOOK_VERSION = {NOTEBOOK_VERSION}")
print(f"QUICK = {QUICK}")

## 0.1 Locate the data

Finds `panel_sindy_ready.csv` wherever you attached it — the working directory and its parents,
anywhere under `/kaggle/input`, or the directory named by the `FINDIST_DATA` environment
variable.

In [ ]:
DATA_NAME = "panel_sindy_ready.csv"


def _walk_dirs(root, max_depth=4):
    """Directories under `root`, breadth-limited, skipping noise."""
    root = Path(root)
    if not root.is_dir():
        return
    skip = {".git", "__pycache__", ".ipynb_checkpoints", "node_modules",
            ".pytest_cache", ".ruff_cache", "site-packages"}
    frontier = [(root, 0)]
    while frontier:
        d, depth = frontier.pop(0)
        yield d
        if depth >= max_depth:
            continue
        try:
            kids = [c for c in d.iterdir() if c.is_dir() and c.name not in skip]
        except OSError:
            continue
        frontier.extend((c, depth + 1) for c in kids)


def find_data():
    """Cheap exact checks first, then a bounded walk of every plausible root."""
    here = Path.cwd().resolve()
    seeds = [here, *here.parents]
    env = os.environ.get("FINDIST_DATA")
    if env:
        seeds.insert(0, Path(env))
    for s in seeds:
        for sub in ("", "notebooks/financialdist", "financialdist", "data"):
            cand = (s / sub if sub else s) / DATA_NAME
            if cand.is_file():
                return cand.resolve()
    for root in (Path("/kaggle/input"), here, Path("/kaggle/working")):
        for d in _walk_dirs(root):
            cand = d / DATA_NAME
            if cand.is_file():
                return cand.resolve()
    raise FileNotFoundError(
        f"{DATA_NAME} not found. Attach it as a Kaggle Dataset (Add Data) or set "
        f"FINDIST_DATA to the directory holding it.")


DATA_PATH = find_data()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd() / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)
print(f"[data] {DATA_PATH}")
print(f"[out ] {OUT_DIR}")

## 0.2 Imports and shared helpers

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

try:
    from sklearn.ensemble import RandomForestRegressor
    HAVE_SKLEARN = True
except Exception as exc:          # pragma: no cover - Kaggle always has it
    HAVE_SKLEARN = False
    print(f"[warn] scikit-learn unavailable ({exc}); the random-forest rung of E2 is skipped")

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 9})
np.set_printoptions(precision=4, suppress=True)

_T_START = time.time()
RESULTS = {"notebook_version": NOTEBOOK_VERSION, "quick": QUICK, "seed": SEED}


def banner(title):
    print("\n" + "=" * 78)
    print(f"  {title}    [t+{time.time() - _T_START:7.1f}s]")
    print("=" * 78)


def show_fig(name):
    """Save the current figure into OUT_DIR/figures and display it."""
    path = FIG_DIR / f"{name}.png"
    plt.gcf().savefig(path, bbox_inches="tight")
    plt.show()
    print(f"[fig] {path}")


def fmt_ci(lo, hi):
    return f"[{lo:+.4f}, {hi:+.4f}]"


def bh_fdr(pvals):
    """Benjamini-Hochberg adjusted p-values."""
    p = np.asarray(pvals, float)
    n = p.size
    order = np.argsort(p)
    adj = np.empty(n)
    running = 1.0
    for rank in range(n - 1, -1, -1):
        running = min(running, p[order[rank]] * n / (rank + 1))
        adj[order[rank]] = running
    return adj

---

# Part 1 — State coordinates, and the boundary fix

**The state.** Twelve asinh-transformed ratios, standardised. `X9 = TL/E` is replaced by its
reciprocal `IEQ = E/TL`, which is bounded near the event and passes smoothly through zero;
`X13`–`X15` are parameters, not states (`DATA_REPORT.md` §3).

**The fix.** `RESEARCH_DESIGN.md` §2 defines the boundary as `S = {x : IEQ(x) = 0}` and calls it a
hyperplane through the origin "because asinh is odd and maps 0 → 0 exactly". That is true of the
`*_t` columns. It is *not* true of the `*_z` columns the file ships, because those subtract the
mean afterwards — which moves the boundary to `IEQ_z = -z_mean/z_std ≈ -0.67`. The cell below
prints both counts. Reading the shipped `IEQ_z < 0` as "distressed" labels **2,580 of 3,564**
firm-years, against a true count of **146**.

So we rebuild the state from `*_t`: every variable is divided by its standard deviation, and
`IEQ` alone is left uncentred. Thresholds stay comparable across equations, and `IEQ_state = 0`
is the boundary, exactly.

One consequence to keep in mind downstream: in the **within** transform the boundary is
firm-specific, sitting at `-mean_i(IEQ)`. Part 7 tracks it per firm rather than assuming zero.

In [ ]:
raw = pd.read_csv(DATA_PATH).sort_values(["SHARE_CODE", "YEAR"]).reset_index(drop=True)

STATE = ["X1_gross_profit", "X2_net_profit_margin", "X3_current_ratio", "X4_quick_ratio",
         "X5_cash_ratio", "X6_receivable_turnover", "X7_inventory_turnover",
         "X8_leverage_asset", "IEQ_equity_to_liability", "X10_dscr",
         "X11_asset_coverage", "X12_mbv"]
SHORT = ["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8", "IEQ", "X10", "X11", "X12"]
D = len(STATE)
I_IEQ = SHORT.index("IEQ")
I_LEV = SHORT.index("X8")

# np.asarray(..., dtype=object) rather than .to_numpy(): pandas 3.0 hands back
# arrow-backed string extension arrays, which do not broadcast like ndarrays.
CODES = np.asarray(raw.SHARE_CODE, dtype=object)
FIRMS = np.asarray(pd.unique(raw.SHARE_CODE), dtype=object)
YEARS = np.unique(np.asarray(raw.YEAR, dtype=int))
N_FIRMS, T = len(FIRMS), len(YEARS)

# The panel is balanced and sorted, so a plain reshape is a valid (firm, year) view.
# Assert it rather than trust it -- every reshape below depends on this.
assert (CODES.reshape(N_FIRMS, T) == FIRMS[:, None]).all()
assert (np.asarray(raw.YEAR, dtype=int).reshape(N_FIRMS, T) == YEARS[None, :]).all()

Tt = raw[[c + "_t" for c in STATE]].to_numpy(float)      # asinh, uncentred
sd = Tt.std(axis=0, ddof=1)
center = Tt.mean(axis=0)
boundary_offset_shipped = float(-center[I_IEQ] / sd[I_IEQ])
center[I_IEQ] = 0.0                                       # <-- the boundary fix

X = ((Tt - center) / sd).reshape(N_FIRMS, T, D)
Y_DIST = raw.Y_distress.to_numpy(float).reshape(N_FIRMS, T)
EVER = raw.ever_distressed.to_numpy(float).reshape(N_FIRMS, T)[:, 0].astype(bool)
IEQ_RAW = raw.IEQ_equity_to_liability.to_numpy(float).reshape(N_FIRMS, T)

banner("Part 1 - state coordinates")
print(f"panel: {N_FIRMS} firms x {T} years = {N_FIRMS * T} firm-years, "
      f"{N_FIRMS * (T - 1)} one-step pairs, {D} states")
print(f"distress firm-years {int(Y_DIST.sum())}, ever-distressed firms {int(EVER.sum())}")
print()
print("boundary diagnostic")
print(f"  shipped IEQ_z puts IEQ = 0 at z = {boundary_offset_shipped:+.4f}, not 0")
print(f"  rows with IEQ < 0                : {int((IEQ_RAW < 0).sum())}")
print(f"  rows with shipped IEQ_z < 0      : "
      f"{int((raw.IEQ_equity_to_liability_z.to_numpy() < 0).sum())}   <- 'distress' if you "
      f"read the sign")
print(f"  rows with rebuilt IEQ_state < 0  : {int((X[:, :, I_IEQ] < 0).sum())}   <- matches")
assert int((X[:, :, I_IEQ] < 0).sum()) == int((IEQ_RAW < 0).sum()), "boundary rebuild failed"

RESULTS["data"] = dict(
    n_firms=N_FIRMS, n_years=T, n_pairs=N_FIRMS * (T - 1),
    distress_firm_years=int(Y_DIST.sum()), ever_distressed=int(EVER.sum()),
    shipped_boundary_offset_z=boundary_offset_shipped,
    rows_ieq_negative=int((IEQ_RAW < 0).sum()),
    rows_shipped_z_negative=int((raw.IEQ_equity_to_liability_z.to_numpy() < 0).sum()),
)

### 1.1 The three diagnostics the design rests on

`RESEARCH_DESIGN.md` §3.1 quotes a between-firm variance share of 0.762, a pooled AR(1) of 0.930
and a within AR(1) of 0.634 for IEQ. Recomputed here so that every later claim sits on numbers
this notebook actually produced.

In [ ]:
def variance_shares(X):
    fm = X.mean(axis=1)                       # (n_firms, d)
    gm = X.reshape(-1, X.shape[2]).mean(axis=0)
    between = (X.shape[1] * ((fm - gm) ** 2)).sum(axis=0)
    total = ((X.reshape(-1, X.shape[2]) - gm) ** 2).sum(axis=0)
    return between / total


def ar1_coeffs(X):
    """Pooled and within-firm scalar AR(1) per variable."""
    a, b = X[:, :-1, :], X[:, 1:, :]
    pooled = np.array([np.polyfit(a[:, :, j].ravel(), b[:, :, j].ravel(), 1)[0]
                       for j in range(X.shape[2])])
    m = X.mean(axis=1, keepdims=True)
    ad, bd = a - m, b - m
    within = np.array([np.polyfit(ad[:, :, j].ravel(), bd[:, :, j].ravel(), 1)[0]
                       for j in range(X.shape[2])])
    return pooled, within


shares = variance_shares(X)
pooled_rho, within_rho = ar1_coeffs(X)
tau = -1.0 / np.log(np.clip(within_rho, 1e-6, 0.999))

banner("Part 1.1 - panel diagnostics")
print(f"{'var':>5} {'between share':>14} {'pooled rho':>11} {'within rho':>11} {'tau (yr)':>9}")
for j in range(D):
    print(f"{SHORT[j]:>5} {shares[j]:14.3f} {pooled_rho[j]:11.3f} "
          f"{within_rho[j]:11.3f} {tau[j]:9.2f}")
print(f"\nIEQ: between share {shares[I_IEQ]:.3f}, pooled {pooled_rho[I_IEQ]:.3f}, "
      f"within {within_rho[I_IEQ]:.3f} (tau = {tau[I_IEQ]:.2f} yr)")
print("design §3.1 quotes 0.762 / 0.930 / 0.634 (2.2 yr)")
print(f"\nDelta t / tau over the twelve states: "
      f"{(1.0 / tau).min():.2f} to {(1.0 / tau).max():.2f}  "
      f"-> O(1), so discrete time only (design §3.3)")

RESULTS["diagnostics"] = dict(
    between_share={SHORT[j]: float(shares[j]) for j in range(D)},
    pooled_ar1={SHORT[j]: float(pooled_rho[j]) for j in range(D)},
    within_ar1={SHORT[j]: float(within_rho[j]) for j in range(D)},
    tau_years={SHORT[j]: float(tau[j]) for j in range(D)},
)

---

# Part 2 — `panel_stlsq`, the core estimator

Discrete-time sparse regression with unit heterogeneity:

$$x_i(t+1) \;=\; \alpha_i + \Theta\big(x_i(t)\big)\,\Xi + \varepsilon_i(t)$$

Two design choices from §4.1, and a note on one of them.

**Fixed effects.** Demeaning $\Theta$ and $Y$ within firm profiles out $\alpha_i$. The design worries
that "demeaning and thresholding interact" and asks for $\alpha_i$ to be profiled *inside* the sparse
loop. It turns out one demeaning at the start is already exactly that: because $\Theta(x)$ does not
depend on $\Xi$, the profiled optimum for any support $S$ is
$\alpha_i^\star = \overline{Y}_i - \overline{\Theta}_{i,S}\,\Xi_S$, and restricting demeaned columns
to $S$ is the same object as demeaning the restricted columns. So the cheap version is the exact
version, and there is no iteration to get wrong.

**Group sparsity.** A term is kept or dropped for all twelve equations jointly, so selection is
`len(terms)` binary decisions rather than `len(terms) × 12`. Thresholds apply to the row norm of the
*column-standardised* coefficients divided by $\sqrt{d}$, which makes the threshold read as
"typical per-equation coefficient" and comparable across libraries. Per-equation STLSQ is available
via `group_sparse=False` for the comparison E2 asks for.

**Libraries.** The constant is omitted everywhere — the firm effects absorb it.

| name | terms | contents |
|---|---|---|
| `linear` | 12 | $x_j$ |
| `lin+sq` | 24 | $+\;x_j^2$ |
| `deg2r` | 45 | $+$ squares $+$ every product involving `IEQ` or `X8` |
| `deg2` | 90 | full quadratic |

`deg2r` is the restricted library of §4.2: the prior is that if anything interacts nonlinearly it
is leverage and the boundary variable. SINDy-PI rational libraries are **not** implemented here —
they need a different regression (implicit, with its own degeneracy handling) and belong in a
follow-up.

In [ ]:
def library_terms(kind, d=D):
    """Index tuples defining a library. The constant is omitted: firm effects absorb it."""
    terms = [(j,) for j in range(d)]
    if kind == "linear":
        pass
    elif kind == "lin+sq":
        terms += [(j, j) for j in range(d)]
    elif kind in ("deg2r", "deg2"):
        key = {I_IEQ, I_LEV}
        for j in range(d):
            for k in range(j, d):
                if kind == "deg2" or j == k or j in key or k in key:
                    terms.append((j, k))
    else:
        raise ValueError(f"unknown library {kind!r}")
    return terms


def term_names(terms):
    out = []
    for t in terms:
        if len(t) == 1:
            out.append(SHORT[t[0]])
        elif t[0] == t[1]:
            out.append(SHORT[t[0]] + "^2")
        else:
            out.append(SHORT[t[0]] + "*" + SHORT[t[1]])
    return out


def theta(Xs, terms):
    """Xs: (n, d) states -> (n, len(terms)) library matrix."""
    out = np.empty((Xs.shape[0], len(terms)))
    for c, t in enumerate(terms):
        out[:, c] = Xs[:, t[0]] if len(t) == 1 else Xs[:, t[0]] * Xs[:, t[1]]
    return out


for _k in ("linear", "lin+sq", "deg2r", "deg2"):
    print(f"{_k:>8}: {len(library_terms(_k)):3d} terms")

In [ ]:
def make_pairs(X, firm_sel=None, t_lo=0, t_hi=None):
    """One-step pairs, rows ordered firm-major then time (so a reshape recovers the panel)."""
    sel = np.arange(X.shape[0]) if firm_sel is None else np.asarray(firm_sel)
    t_hi = X.shape[1] - 1 if t_hi is None else t_hi
    ts = np.arange(t_lo, t_hi)
    Xt = X[np.ix_(sel, ts)].reshape(-1, X.shape[2])
    Xn = X[np.ix_(sel, ts + 1)].reshape(-1, X.shape[2])
    return Xt, Xn, len(sel)


def within(M, nf):
    """Firm-demean. Relies on the balanced layout make_pairs produces."""
    assert M.shape[0] % nf == 0, "unbalanced block layout"
    V = M.reshape(nf, -1, M.shape[1])
    m = V.mean(axis=1)
    return (V - m[:, None, :]).reshape(M.shape), m


def panel_stlsq(Th, Yn, nf, threshold, ridge=1e-3,
                fixed_effects=True, group_sparse=True, max_iter=20):
    """
    Discrete-time sparse regression with unit heterogeneity.

    fixed_effects : demean Theta and Y within firm, which profiles out alpha_i exactly
                    (see the note above). False gives a pooled fit with one common
                    intercept -- the cautionary baseline of design §3.1.
    group_sparse  : keep or drop each term for all d equations jointly. Thresholds act on
                    the row norm of the column-standardised coefficients over sqrt(d).

    Returns Xi in the original library scale, so coefficients are comparable across fits.
    """
    d = Yn.shape[1]
    if fixed_effects:
        A, thmean = within(Th, nf)
        B, ymean = within(Yn, nf)
    else:
        gt, gy = Th.mean(axis=0, keepdims=True), Yn.mean(axis=0, keepdims=True)
        A, B = Th - gt, Yn - gy
        thmean, ymean = np.repeat(gt, nf, 0), np.repeat(gy, nf, 0)

    sc = A.std(axis=0, ddof=0)
    sc[sc <= 1e-12] = 1.0
    A = A / sc
    n, p = A.shape
    G0, R = A.T @ A, A.T @ B
    lam = ridge * n

    def solve(mask):
        C = np.zeros((p, d))
        idx = np.flatnonzero(mask)
        if idx.size:
            C[idx] = np.linalg.solve(G0[np.ix_(idx, idx)] + lam * np.eye(idx.size), R[idx])
        return C

    if group_sparse:
        act = np.ones(p, bool)
        for _ in range(max_iter):
            C = solve(act)
            new = (np.linalg.norm(C, axis=1) / math.sqrt(d)) >= threshold
            if np.array_equal(new, act):
                break
            act = new
            if not act.any():
                break
        C = solve(act)
        C[~act] = 0.0
    else:
        def solve_cols(mask):
            C = np.zeros((p, d))
            for q in range(d):
                idx = np.flatnonzero(mask[:, q])
                if idx.size:
                    C[idx, q] = np.linalg.solve(
                        G0[np.ix_(idx, idx)] + lam * np.eye(idx.size), R[idx, q])
            return C

        mask = np.ones((p, d), bool)
        for _ in range(max_iter):
            new = np.abs(solve_cols(mask)) >= threshold
            if np.array_equal(new, mask):
                break
            mask = new
            if not mask.any():
                break
        C = solve_cols(mask)          # always refit on the support we are returning

    Xi = C / sc[:, None]
    active = np.abs(Xi).sum(axis=1) > 0
    return dict(Xi=Xi, Xi_scaled=C, alpha=ymean - thmean @ Xi,
                active=active, n_terms=int(active.sum()))

### 2.1 Evaluation — rollouts, and the firm effect for held-out firms

A firm-blocked split has a problem the row-blocked splits in the literature never face: **a
held-out firm has no estimated $\alpha_i$.** Three ways out, and they are not equivalent:

1. demean the test firm using its own full sample — no *dynamics* leak, but the firm's level over
   the evaluation window has been seen. This is what §3.2's table does;
2. read $\alpha_i$ off a burn-in window and evaluate strictly after it — nothing leaks;
3. drop the firm effect at test time — measures something else entirely.

This notebook uses **(2)**: $\alpha_i$ comes from the pairs inside 2013–2018, and every rollout
starts at index `T_BURN - 1` or later. It costs half the panel and it is the only one of the three
that can be reported without a caveat.

Selection is on **horizon-3 rollout RMSE**, not one-step $R^2$ (§4.2) — one-step error is dominated
by the persistent component, and a model can win it by doing nothing. Rollouts are clipped to a box
of $\pm 8$ z-units; the clipping rate is reported, because a quadratic map that only looks good
because it was clipped is not a good map.

In [ ]:
def alpha_burnin(X, sel, Xi, terms, t_end=T_BURN):
    """Firm effects for held-out firms, read off their first t_end observations only."""
    Xt, Xn, nf = make_pairs(X, sel, 0, t_end - 1)
    res = (Xn - theta(Xt, terms) @ Xi).reshape(nf, -1, X.shape[2])
    return res.mean(axis=1)


def rollout_errors(X, sel, Xi, alpha, terms, horizons=HORIZONS,
                   t0_min=T_BURN - 1, clip=CLIP):
    """Per-firm mean squared error at each horizon, and the fraction of clipped entries."""
    sel = np.asarray(sel)
    Tn = X.shape[1]
    per_firm, clipped, total = {}, 0, 0
    for h in horizons:
        errs = []
        for s in range(t0_min, Tn - h):
            x = X[sel, s, :].copy()
            for _ in range(h):
                x = alpha + theta(x, terms) @ Xi
                clipped += int(np.count_nonzero(np.abs(x) > clip))
                total += x.size
                np.clip(x, -clip, clip, out=x)
            errs.append((x - X[sel, s + h, :]) ** 2)
        per_firm[h] = (np.stack(errs).mean(axis=(0, 2)) if errs
                       else np.full(len(sel), np.nan))
    return per_firm, (clipped / total if total else 0.0)


def rmse_from(per_firm):
    return {h: float(np.sqrt(np.nanmean(v))) for h, v in per_firm.items()}


def firm_folds(n, k, rng):
    idx = rng.permutation(n)
    return [np.sort(idx[i::k]) for i in range(k)]


def fit_and_score(X, tr, te, lib, threshold, ridge, fixed_effects=True, group_sparse=True,
                  horizons=HORIZONS, alpha_window=T_BURN):
    """
    Fit on training firms, score rollouts on held-out firms.

    alpha_window : how many of the test firm's own observations alpha_i may see. The default
                   T_BURN is leakage-free. Passing T is the *oracle* variant -- it lets alpha_i
                   see the evaluation window, so it is an upper bound, not a result.
    """
    terms = library_terms(lib)
    Xt, Xn, nf = make_pairs(X, tr)
    fit = panel_stlsq(theta(Xt, terms), Xn, nf, threshold, ridge,
                      fixed_effects=fixed_effects, group_sparse=group_sparse)
    if fixed_effects:
        a_te = alpha_burnin(X, te, fit["Xi"], terms, t_end=alpha_window)
    else:
        a_te = np.repeat(fit["alpha"][:1], len(te), axis=0)   # one common intercept
    per_firm, clipped = rollout_errors(X, te, fit["Xi"], a_te, terms, horizons)
    return fit, per_firm, clipped

---

# Part 3 — Synthetic recovery test (**the estimator gate**)

§10: *"If the estimator cannot recover a known map under exactly this design, no result on the
real data means anything."*

Ground truth is a sparse quadratic discrete map in twelve dimensions with

* diagonal $\rho$ set to the empirical within-firm AR(1) coefficients (§3.3),
* six random off-diagonal linear couplings,
* eight quadratic terms at $\pm 0.10$,
* firm intercepts calibrated so the between-firm variance share matches the empirical 0.76,
* noise scaled so the stationary within-firm variance is 1, matching standardised data,

sampled at exactly $N = 324$, $T = 11$. Support is compared at **term level** (the union across
equations), which is what group sparsity selects. The calibration formula for the intercept spread
follows from the stationary AR(1) decomposition: with within variance 1,
$\sigma_\alpha = (1-\rho)\sqrt{s/(1-s)}$.

**Pass condition: support F1 ≥ 0.8 at some threshold.** Anything less and Parts 4–7 are reporting
estimator noise.

In [ ]:
def make_synth(rng, n_firms=N_FIRMS, T_=T, d=D, between_share=0.76,
               n_quad=8, quad_scale=0.06, spin=60, rho=None, box=12.0):
    """A known sparse quadratic map, calibrated to this panel's N, T, noise and heterogeneity."""
    rho = np.clip(within_rho[:d], 0.05, 0.95) if rho is None else np.asarray(rho)
    A = np.diag(rho)
    for o in rng.choice(d * d, size=6, replace=False):
        j, k = divmod(int(o), d)
        if j != k:
            A[j, k] = rng.uniform(-0.08, 0.08)

    terms = library_terms("deg2", d)
    Xi_true = np.zeros((len(terms), d))
    Xi_true[:d, :] = A.T                       # x_next[:, q] = sum_j A[q, j] x_j
    for c in rng.choice(np.arange(d, len(terms)), size=n_quad, replace=False):
        Xi_true[int(c), int(rng.integers(0, d))] = quad_scale * rng.choice([-1.0, 1.0])

    sig = np.sqrt(1.0 - rho ** 2)
    sig_a = (1.0 - rho) * math.sqrt(between_share / (1.0 - between_share))
    a = rng.normal(0.0, sig_a, size=(n_firms, d))

    x = rng.normal(0.0, 1.0, size=(n_firms, d))
    traj, n_clipped, n_total = [], 0, 0
    for t in range(spin + T_):
        if t >= spin:
            traj.append(x.copy())
        x = a + theta(x, terms) @ Xi_true + rng.normal(0.0, sig, size=(n_firms, d))
        n_clipped += int(np.count_nonzero(np.abs(x) > box))
        n_total += x.size
        np.clip(x, -box, box, out=x)
    return np.stack(traj, axis=1), Xi_true, terms, n_clipped / n_total


if RUN_SYNTH:
    banner("Part 3 - synthetic recovery test (gate)")
    rng = np.random.default_rng(SEED)
    Xsyn, Xi_true, syn_terms, syn_clip = make_synth(rng)
    true_active = np.abs(Xi_true).sum(axis=1) > 0

    print(f"generated {Xsyn.shape[0]} firms x {Xsyn.shape[1]} years x {Xsyn.shape[2]} states")
    print(f"  true active terms      : {int(true_active.sum())} of {len(syn_terms)}")
    print(f"  realised between share : {variance_shares(Xsyn).mean():.3f} (target 0.76)")
    print(f"  max |state|            : {np.abs(Xsyn).max():.2f}")
    print(f"  clipped during sim     : {syn_clip:.4%}  "
          f"(must stay near zero, or the 'known map' is not the map that generated the data)")

    Xt, Xn, nf = make_pairs(Xsyn)
    Th = theta(Xt, syn_terms)
    syn_rows = []
    for thr in np.logspace(-3, -0.5, 14):
        f = panel_stlsq(Th, Xn, nf, float(thr), ridge=1e-3)
        pred = f["active"]
        tp = int((pred & true_active).sum())
        fp = int((pred & ~true_active).sum())
        fn = int((~pred & true_active).sum())
        prec = tp / max(tp + fp, 1)
        rec = tp / max(tp + fn, 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-12)
        err = np.abs(f["Xi"][true_active] - Xi_true[true_active])
        syn_rows.append(dict(threshold=float(thr), n_terms=f["n_terms"],
                             precision=prec, recall=rec, f1=f1,
                             mae_coef=float(err.mean())))

    syn = pd.DataFrame(syn_rows)
    print()
    print(syn.to_string(index=False,
                        formatters={"threshold": "{:.4f}".format,
                                    "precision": "{:.3f}".format,
                                    "recall": "{:.3f}".format,
                                    "f1": "{:.3f}".format,
                                    "mae_coef": "{:.4f}".format}))

    best = syn.loc[syn.f1.idxmax()]
    SYNTH_PASS = bool(best.f1 >= 0.8)
    print(f"\nbest support F1 = {best.f1:.3f} at threshold {best.threshold:.4f} "
          f"({int(best.n_terms)} terms, coefficient MAE {best.mae_coef:.4f})")
    print("VERDICT: " + ("PASS - the estimator recovers a known map at this N, T and noise."
                         if SYNTH_PASS else
                         "FAIL - fix the estimator before reading anything below."))

    RESULTS["synthetic"] = dict(table=syn_rows, best_f1=float(best.f1),
                                best_threshold=float(best.threshold), passed=SYNTH_PASS,
                                clip_rate=float(syn_clip),
                                between_share=float(variance_shares(Xsyn).mean()))

    fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4))
    ax[0].semilogx(syn.threshold, syn.precision, "o-", label="precision")
    ax[0].semilogx(syn.threshold, syn.recall, "s-", label="recall")
    ax[0].semilogx(syn.threshold, syn.f1, "^-", lw=2, label="F1")
    ax[0].axhline(0.8, color="k", ls=":", lw=0.8)
    ax[0].set_xlabel("threshold"); ax[0].set_ylabel("term-level support")
    ax[0].set_title("Support recovery, known map"); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].semilogx(syn.threshold, syn.mae_coef, "o-", color="C3")
    ax[1].set_xlabel("threshold"); ax[1].set_ylabel("MAE on true coefficients")
    ax[1].set_title("Coefficient bias"); ax[1].grid(alpha=0.3)
    plt.tight_layout()
    show_fig("synthetic_recovery")
else:
    SYNTH_PASS = None
    print("[skip] RUN_SYNTH = False")

---

# Part 4 — E1, the gate experiment

**H:** there exists a threshold/library combination where degree-2 terms beat AR(1) on horizon-3
rollout in the within transform.
**Falsified if:** no degree-2 configuration dominates AR(1) anywhere on the frontier.

Two distinct numbers come out of this cell, and conflating them would be the easiest mistake in
the project:

* the **nested-CV estimate** — threshold, library and ridge chosen on inner folds, evaluated once
  on held-out outer firms. This is the honest headline;
* the **Pareto frontier** — every configuration scored directly on the outer test firms. This is a
  *descriptive* curve for reading where the knee is. It is optimistic by construction and is
  labelled as such on the figure.

Baselines overlaid: persistence, AR(1) with firm effects, and panel-VAR(1) with firm effects.

The gate verdict is decided by a **paired fold-by-fold comparison**, not by whether one mean lands
under another. On this data the best nonlinear mean can sit 3 × 10⁻⁴ below AR(1), which is noise;
a verdict built on that would be an artifact of rounding. Each library is compared at its best
average configuration, fold by fold, and has to clear zero by two standard errors. Five folds make
that a weak test — the interval is printed so the weakness is visible rather than hidden.

In [ ]:
def fit_ar1(X, tr):
    """Scalar AR(1) per variable with firm effects, expressed in the linear library."""
    terms = library_terms("linear")
    Xt, Xn, nf = make_pairs(X, tr)
    A, thmean = within(Xt, nf)
    B, ymean = within(Xn, nf)
    Xi = np.zeros((D, D))
    for j in range(D):
        Xi[j, j] = float(A[:, j] @ B[:, j] / max(A[:, j] @ A[:, j], 1e-12))
    return dict(Xi=Xi, alpha=ymean - thmean @ Xi, n_terms=D), terms


def score_ar1(X, tr, te, horizons=HORIZONS):
    fit, terms = fit_ar1(X, tr)
    a_te = alpha_burnin(X, te, fit["Xi"], terms)
    per_firm, clipped = rollout_errors(X, te, fit["Xi"], a_te, terms, horizons)
    return fit, per_firm, clipped


def score_persistence(X, te, horizons=HORIZONS):
    terms = library_terms("linear")
    Xi = np.eye(D)
    a = np.zeros((len(te), D))
    return rollout_errors(X, te, Xi, a, terms, horizons)


if RUN_E1:
    banner("Part 4 - E1 library / threshold sweep")
    thr_grid = np.logspace(THRESHOLD_GRID[0], THRESHOLD_GRID[1], int(THRESHOLD_GRID[2]))
    CONFIGS = [(lib, float(t), float(al))
               for lib in LIBRARIES for t in thr_grid for al in ALPHA_GRID]
    print(f"{len(CONFIGS)} configurations x {N_OUTER} outer x {N_INNER} inner folds")

    rng = np.random.default_rng(SEED)
    outer = firm_folds(N_FIRMS, N_OUTER, rng)
    nested_rows, frontier_rows = [], []

    for o, te in enumerate(outer):
        tr = np.setdiff1d(np.arange(N_FIRMS), te)
        inner = firm_folds(len(tr), N_INNER, rng)
        inner_score = np.zeros(len(CONFIGS))
        for i, ite_loc in enumerate(inner):
            ite = tr[ite_loc]
            itr = np.setdiff1d(tr, ite)
            for c, (lib, thrv, al) in enumerate(CONFIGS):
                _, pf, _ = fit_and_score(X, itr, ite, lib, thrv, al, horizons=(H_SELECT,))
                inner_score[c] += np.sqrt(np.nanmean(pf[H_SELECT])) / N_INNER
        bi = int(np.argmin(inner_score))
        lib, thrv, al = CONFIGS[bi]

        fit, pf, clipped = fit_and_score(X, tr, te, lib, thrv, al)
        r = rmse_from(pf)
        nested_rows.append(dict(fold=o, library=lib, threshold=thrv, ridge=al,
                                n_terms=fit["n_terms"], clipped=clipped,
                                **{f"rmse_h{h}": r[h] for h in HORIZONS}))
        print(f"  fold {o}: selected {lib:>7} thr={thrv:.4f} ridge={al:.0e} "
              f"-> {fit['n_terms']:3d} terms, h3 RMSE {r[H_SELECT]:.4f}, "
              f"clipped {clipped:.2%}")

        # descriptive frontier: every config scored on this fold's held-out firms
        for lib_, thrv_, al_ in CONFIGS:
            f2, pf2, cl2 = fit_and_score(X, tr, te, lib_, thrv_, al_, horizons=(H_SELECT,))
            frontier_rows.append(dict(fold=o, library=lib_, threshold=thrv_, ridge=al_,
                                      n_terms=f2["n_terms"], clipped=cl2,
                                      rmse=float(np.sqrt(np.nanmean(pf2[H_SELECT])))))

        for name, sc in (("AR(1)+FE", score_ar1(X, tr, te)[1:]),
                         ("persistence", score_persistence(X, te))):
            frontier_rows.append(dict(fold=o, library=name, threshold=np.nan, ridge=np.nan,
                                      n_terms=D, clipped=sc[1],
                                      rmse=float(np.sqrt(np.nanmean(sc[0][H_SELECT])))))
        f3, pf3, cl3 = fit_and_score(X, tr, te, "linear", 0.0, 1e-4, horizons=(H_SELECT,))
        frontier_rows.append(dict(fold=o, library="panelVAR+FE", threshold=np.nan,
                                  ridge=1e-4, n_terms=f3["n_terms"], clipped=cl3,
                                  rmse=float(np.sqrt(np.nanmean(pf3[H_SELECT])))))

    nested = pd.DataFrame(nested_rows)
    frontier = pd.DataFrame(frontier_rows)
    print("\nnested-CV estimate (the honest headline):")
    print(nested.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print(f"\nmean held-out horizon-{H_SELECT} RMSE = "
          f"{nested[f'rmse_h{H_SELECT}'].mean():.4f} "
          f"+- {nested[f'rmse_h{H_SELECT}'].std(ddof=1):.4f} (sd over folds)")

    E1_BEST = nested.library.mode().iloc[0]
    _sub = nested[nested.library == E1_BEST]
    E1_THR = float(np.exp(np.log(_sub.threshold).mean()))
    E1_RIDGE = float(np.exp(np.log(_sub.ridge).mean()))
    print(f"modal selection: {E1_BEST}, geometric-mean threshold {E1_THR:.4f}, "
          f"ridge {E1_RIDGE:.1e}  (used as the fixed configuration in E3 and E5)")

    RESULTS["E1"] = dict(nested=nested_rows,
                         mean_rmse_h3=float(nested[f"rmse_h{H_SELECT}"].mean()),
                         sd_rmse_h3=float(nested[f"rmse_h{H_SELECT}"].std(ddof=1)),
                         selected_library=E1_BEST, selected_threshold=E1_THR,
                         selected_ridge=E1_RIDGE)
else:
    E1_BEST, E1_THR, E1_RIDGE = "deg2", 0.05, 1e-3
    frontier = None
    print(f"[skip] RUN_E1 = False; falling back to {E1_BEST} / {E1_THR} for E3 and E5")

In [ ]:
if RUN_E1:
    agg = (frontier.groupby(["library", "threshold", "ridge"], dropna=False)
                   .agg(n_terms=("n_terms", "mean"), rmse=("rmse", "mean"))
                   .reset_index())
    plt.figure(figsize=(6.4, 4.4))
    for lib in LIBRARIES:
        # best RMSE achieved at each term count, across thresholds and ridge values
        s = (agg[agg.library == lib]
             .groupby("n_terms", as_index=False).rmse.min().sort_values("n_terms"))
        plt.plot(s.n_terms, s.rmse, "o-", ms=3.5, label=lib)
    for name, style in (("AR(1)+FE", "k--"), ("panelVAR+FE", "b-."), ("persistence", "r:")):
        s = agg[agg.library == name]
        if len(s):
            plt.axhline(float(s.rmse.mean()), ls=style[1:], color=style[0], lw=1.2,
                        label=f"{name} ({float(s.rmse.mean()):.3f})")
    plt.xlabel("active terms"); plt.ylabel(f"held-out horizon-{H_SELECT} rollout RMSE")
    plt.title("E1 Pareto frontier (descriptive - scored on the outer test firms)")
    plt.legend(fontsize=7.5); plt.grid(alpha=0.3)
    plt.tight_layout()
    show_fig("E1_pareto")

    # A bare "is the mean below the bar" comparison is not a gate: on this data the best
    # nonlinear mean can sit 3e-4 under AR(1), which is noise. Compare fold by fold instead,
    # at each library's best average configuration, and require the paired interval to clear
    # zero. Five folds is a weak test and the interval says so -- that is the point.
    ar1_fold = (frontier[frontier.library == "AR(1)+FE"]
                .set_index("fold").rmse.sort_index())
    ar1_bar = float(ar1_fold.mean())
    gate_rows = []
    for lib in LIBRARIES:
        s = frontier[frontier.library == lib]
        m = s.groupby(["threshold", "ridge"]).rmse.mean()
        thr_, rg_ = m.idxmin()
        sub = s[(s.threshold == thr_) & (s.ridge == rg_)].set_index("fold").sort_index()
        delta = (sub.rmse - ar1_fold).to_numpy()
        se = float(delta.std(ddof=1) / math.sqrt(len(delta)))
        gate_rows.append(dict(library=lib, threshold=float(thr_), ridge=float(rg_),
                              n_terms=float(sub.n_terms.mean()), rmse=float(m.min()),
                              delta_vs_ar1=float(delta.mean()), se=se,
                              lo=float(delta.mean() - 2 * se),
                              hi=float(delta.mean() + 2 * se),
                              beats=bool(delta.mean() + 2 * se < 0)))
    gate = pd.DataFrame(gate_rows)
    print(f"AR(1)+FE bar: {ar1_bar:.4f}   (per fold: "
          + ", ".join(f"{v:.4f}" for v in ar1_fold) + ")")
    print("\nbest configuration per library, paired against AR(1)+FE fold by fold:")
    print(gate.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

    winners = gate[(gate.library != "linear") & gate.beats]
    if len(winners):
        print("\nE1 GATE PASSED: " + ", ".join(winners.library)
              + " beats AR(1)+FE with a paired interval clear of zero.")
    else:
        margin = gate[gate.library != "linear"].delta_vs_ar1.min()
        print(f"\nE1 GATE NOT PASSED: best nonlinear library is {margin:+.4f} against AR(1)+FE, "
              f"interval spans zero. Design §7 row 3 or 4 - the methods-forward narrative.")
    RESULTS["E1"]["ar1_bar"] = ar1_bar
    RESULTS["E1"]["gate"] = gate_rows
    RESULTS["E1"]["gate_passed"] = bool(len(winners) > 0)
    # The whole descriptive frontier, averaged over folds -- every configuration scored, so an
    # appendix can reproduce the figure without rerunning the sweep.
    RESULTS["E1"]["frontier"] = (agg.sort_values(["library", "threshold", "ridge"])
                                    .to_dict("records"))
    RESULTS["E1"]["ar1_bar_per_fold"] = [float(v) for v in ar1_fold]

---

# Part 5 — E2, the baseline ladder

Persistence → AR(1)+FE → VAR(1) pooled → panel-VAR(1)+FE → linear SINDy → degree-2 group-sparse
SINDy → per-equation STLSQ → random forest.

One extra rung that is **not** a competitor: *panel-VAR(1)+FE, oracle α*. It is the same model with
$\alpha_i$ read off the test firm's whole sample instead of the burn-in window, so it sees the
evaluation period and cannot be reported as a result. It is here to separate two explanations of
any gap between the fixed-effects models and the pooled ones — a worse *map*, or a noisier
*intercept* estimated from five pairs. With 76% of variance sitting between firms, the second is
not a small effect, and the ladder is misread without it.

20 firm-blocked splits, held-out horizon-1/3/5 rollout RMSE, and **paired bootstrap CIs on the
delta against AR(1)+FE**, resampled over firms rather than over splits — firms are the independent
unit here, splits are not.

The point of this table is to show SINDy's position honestly, including where it loses. A sparse
model that ties a random forest at a fraction of the parameter count is a good result and should
be framed as one. The LSTM rung of §5 E2 is omitted: it needs a deep-learning dependency for a rung
that the design itself only asks for as a ceiling reference.

In [ ]:
def score_rf(X, tr, te, horizons=HORIZONS, seed=0):
    """Random forest on within-demeaned states. Test firms are demeaned on the burn-in window."""
    Xt, Xn, nf = make_pairs(X, tr)
    A, _ = within(Xt, nf)
    B, _ = within(Xn, nf)
    rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1,
                               random_state=seed)
    rf.fit(A, B)
    te = np.asarray(te)
    m = X[te, :T_BURN, :].mean(axis=1)                    # burn-in level, no leakage
    per_firm = {}
    for h in horizons:
        errs = []
        for s in range(T_BURN - 1, X.shape[1] - h):
            x = X[te, s, :].copy()
            for _ in range(h):
                x = np.clip(rf.predict(x - m) + m, -CLIP, CLIP)
            errs.append((x - X[te, s + h, :]) ** 2)
        per_firm[h] = np.stack(errs).mean(axis=(0, 2))
    return per_firm


if RUN_E2:
    banner("Part 5 - E2 baseline ladder")
    # The nonlinear rungs are always deg2, whatever E1 selected -- otherwise a run in which
    # E1 picks "linear" produces a ladder with three identical rows and no nonlinear rung.
    names = ["persistence", "AR(1)+FE", "VAR(1) pooled", "panel-VAR(1)+FE", "linear SINDy",
             "deg2 group-sparse", "deg2 per-equation", "panel-VAR(1)+FE, oracle alpha"]
    if HAVE_SKLEARN:
        names.append("random forest")

    # firm -> list of per-firm MSE at H_SELECT, one entry per split the firm was held out in
    acc = {n: [[] for _ in range(N_FIRMS)] for n in names}
    rows = {n: {h: [] for h in HORIZONS} for n in names}
    nterms = {n: [] for n in names}

    for seed in range(N_SEEDS_E2):
        rng = np.random.default_rng(1000 + seed)
        perm = rng.permutation(N_FIRMS)
        te = np.sort(perm[: int(0.3 * N_FIRMS)])
        tr = np.sort(perm[int(0.3 * N_FIRMS):])

        got = {}
        got["persistence"] = (score_persistence(X, te)[0], D)
        f, pf, _ = score_ar1(X, tr, te); got["AR(1)+FE"] = (pf, D)
        f, pf, _ = fit_and_score(X, tr, te, "linear", 0.0, 1e-4, fixed_effects=False)
        got["VAR(1) pooled"] = (pf, f["n_terms"])
        f, pf, _ = fit_and_score(X, tr, te, "linear", 0.0, 1e-4)
        got["panel-VAR(1)+FE"] = (pf, f["n_terms"])
        f, pf, _ = fit_and_score(X, tr, te, "linear", E1_THR, E1_RIDGE)
        got["linear SINDy"] = (pf, f["n_terms"])
        f, pf, _ = fit_and_score(X, tr, te, "deg2", E1_THR, E1_RIDGE)
        got["deg2 group-sparse"] = (pf, f["n_terms"])
        f, pf, _ = fit_and_score(X, tr, te, "deg2", E1_THR, E1_RIDGE, group_sparse=False)
        got["deg2 per-equation"] = (pf, f["n_terms"])
        f, pf, _ = fit_and_score(X, tr, te, "linear", 0.0, 1e-4, alpha_window=T)
        got["panel-VAR(1)+FE, oracle alpha"] = (pf, f["n_terms"])
        if HAVE_SKLEARN:
            got["random forest"] = (score_rf(X, tr, te, seed=seed), np.nan)

        for n in names:
            pf, nt = got[n]
            nterms[n].append(nt)
            for h in HORIZONS:
                rows[n][h].append(float(np.sqrt(np.nanmean(pf[h]))))
            for k, firm in enumerate(te):
                acc[n][firm].append(float(pf[H_SELECT][k]))
        print(f"  seed {seed:2d} done  [t+{time.time() - _T_START:6.1f}s]")

    # paired bootstrap over firms, on the per-firm mean squared error at H_SELECT
    firm_mse = {n: np.array([np.mean(v) if v else np.nan for v in acc[n]]) for n in names}
    ok = np.all(np.stack([np.isfinite(firm_mse[n]) for n in names]), axis=0)
    rb = np.random.default_rng(7)
    idx_ok = np.flatnonzero(ok)
    boot = {n: [] for n in names}
    for _ in range(2000):
        pick = rb.choice(idx_ok, size=idx_ok.size, replace=True)
        for n in names:
            boot[n].append(np.sqrt(firm_mse[n][pick].mean()))
    boot = {n: np.array(v) for n, v in boot.items()}

    e2_rows = []
    for n in names:
        delta = boot[n] - boot["AR(1)+FE"]
        lo, hi = np.percentile(delta, [2.5, 97.5])
        finite = [v for v in nterms[n] if np.isfinite(v)]
        e2_rows.append(dict(
            model=n, n_terms=float(np.mean(finite)) if finite else float("nan"),
            **{f"rmse_h{h}": float(np.mean(rows[n][h])) for h in HORIZONS},
            d_vs_ar1=float(np.mean(delta)), ci_lo=float(lo), ci_hi=float(hi),
            beats_ar1=bool(hi < 0)))
    e2 = pd.DataFrame(e2_rows)
    print()
    print(e2.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print(f"\n(delta and CI are on horizon-{H_SELECT} RMSE against AR(1)+FE; "
          f"paired bootstrap over {idx_ok.size} firms, 2000 draws. Negative = better.)")
    RESULTS["E2"] = dict(table=e2_rows, n_seeds=N_SEEDS_E2, n_firms_bootstrap=int(idx_ok.size))

    plt.figure(figsize=(7.0, 3.6))
    o = np.argsort(e2[f"rmse_h{H_SELECT}"].to_numpy())
    plt.barh(np.arange(len(o)), e2[f"rmse_h{H_SELECT}"].to_numpy()[o], color="C0", alpha=0.8)
    plt.yticks(np.arange(len(o)), e2.model.to_numpy()[o], fontsize=8)
    plt.axvline(float(e2.loc[e2.model == "AR(1)+FE", f"rmse_h{H_SELECT}"].iloc[0]),
                color="k", ls="--", lw=1, label="AR(1)+FE")
    plt.xlabel(f"held-out horizon-{H_SELECT} rollout RMSE")
    plt.title(f"E2 baseline ladder ({N_SEEDS_E2} firm-blocked splits)")
    plt.legend(fontsize=8); plt.grid(alpha=0.3, axis="x")
    plt.tight_layout()
    show_fig("E2_ladder")
else:
    print("[skip] RUN_E2 = False")

---

# Part 6 — E3, cohort contrast

**H:** the 28 ever-distressed firms occupy a structurally different vector field from the other 296.
**Falsified if:** $p > 0.10$ — which is itself a publishable finding, because it says distress is a
matter of initial condition and noise rather than of distinct mechanism.

Test statistic $T = \lVert \hat\Xi_D - \hat\Xi_H \rVert_F$, with a **permutation null** that
re-assigns the cohort label across firms while preserving cohort sizes. That null is doing real
work: a 28-firm fit is much noisier than a 296-firm fit, and a naive comparison would read that
asymmetry as structure. Per-term statistics get two-sided permutation p-values under
Benjamini–Hochberg control at FDR 0.10.

Alongside it, E-SINDy inclusion probabilities from 500 bootstraps over firms within each cohort —
the design's secondary metric, and the one that speaks directly to "which terms are supported by
the data".

In [ ]:
if RUN_E3:
    banner("Part 6 - E3 cohort contrast")
    terms3 = library_terms(E1_BEST)
    names3 = term_names(terms3)
    idx_all = np.arange(N_FIRMS)
    n_d = int(EVER.sum())
    print(f"library {E1_BEST} ({len(terms3)} terms), threshold {E1_THR:.4f}, "
          f"ridge {E1_RIDGE:.1e}; cohorts {n_d} / {N_FIRMS - n_d}")

    def cohort_fit(sel):
        Xt, Xn, nf = make_pairs(X, sel)
        return panel_stlsq(theta(Xt, terms3), Xn, nf, E1_THR, E1_RIDGE)

    fit_d = cohort_fit(idx_all[EVER])
    fit_h = cohort_fit(idx_all[~EVER])
    diff = fit_d["Xi"] - fit_h["Xi"]
    T_obs = float(np.linalg.norm(diff))
    t_obs = np.linalg.norm(diff, axis=1)
    print(f"observed T = ||Xi_D - Xi_H||_F = {T_obs:.4f}   "
          f"({fit_d['n_terms']} vs {fit_h['n_terms']} active terms)")

    rng = np.random.default_rng(SEED + 11)
    T_null = np.empty(N_PERM_E3)
    t_null_ge = np.zeros(len(terms3))
    for b in range(N_PERM_E3):
        perm = rng.permutation(N_FIRMS)
        fd = cohort_fit(perm[:n_d])
        fh = cohort_fit(perm[n_d:])
        dd = fd["Xi"] - fh["Xi"]
        T_null[b] = np.linalg.norm(dd)
        t_null_ge += (np.linalg.norm(dd, axis=1) >= t_obs)
        if (b + 1) % max(N_PERM_E3 // 5, 1) == 0:
            print(f"  {b + 1}/{N_PERM_E3} permutations  [t+{time.time() - _T_START:6.1f}s]")

    p_T = float((1 + int((T_null >= T_obs).sum())) / (1 + N_PERM_E3))
    p_term = (1 + t_null_ge) / (1 + N_PERM_E3)
    q_term = bh_fdr(p_term)
    print(f"\npermutation p on T: {p_T:.4f}  "
          f"(null mean {T_null.mean():.4f}, 95th pct {np.percentile(T_null, 95):.4f})")

    # E-SINDy inclusion probabilities, bootstrapping firms within each cohort
    def inclusion(sel, n_boot):
        sel = np.asarray(sel)
        hits = np.zeros(len(terms3))
        rg = np.random.default_rng(SEED + 23)
        for _ in range(n_boot):
            hits += cohort_fit(rg.choice(sel, size=sel.size, replace=True))["active"]
        return hits / n_boot

    inc_d = inclusion(idx_all[EVER], N_BOOT_E3)
    inc_h = inclusion(idx_all[~EVER], N_BOOT_E3)

    e3 = pd.DataFrame(dict(term=names3, t_stat=t_obs, p=p_term, q_bh=q_term,
                           incl_distressed=inc_d, incl_healthy=inc_h,
                           d_incl=inc_d - inc_h)).sort_values("p")
    sig = e3[e3.q_bh < 0.10]
    print(f"\nterms surviving BH FDR < 0.10: {len(sig)}")
    print(e3.head(15).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    verdict = ("E3: cohorts differ structurally (design §7, application-forward narrative)."
               if p_T <= 0.10 else
               "E3: no structural difference. Distress is initial condition + noise - "
               "design §7 row 2 or 4, and a finding in its own right.")
    print("\n" + verdict)
    RESULTS["E3"] = dict(library=E1_BEST, threshold=E1_THR, T_obs=T_obs, p_T=p_T,
                         n_perm=N_PERM_E3, n_boot=N_BOOT_E3,
                         null_mean=float(T_null.mean()),
                         null_p95=float(np.percentile(T_null, 95)),
                         n_terms_fdr10=int(len(sig)),
                         terms=e3.to_dict("records"), verdict=verdict)
else:
    print("[skip] RUN_E3 = False")

---

# Part 7 — E5, first-passage validation

**H:** the fitted stochastic map reproduces the observed distribution of crossing times.
**Falsified if:** it fails to beat ranking firms by their current IEQ. That trivial rule is the
honest bar and it is a strong one.

Protocol: fit on 2013–2018, roll each firm forward through 2019–2023 with residuals resampled as
**whole rows** (which preserves the cross-equation covariance $\hat\Sigma_\varepsilon$ exactly,
rather than assuming it is Gaussian), and record whether the path enters distress —
`IEQ < 0` in two consecutive years. Rollouts are in level coordinates ($\alpha_i$ is added back at
every step), so after Part 1 the crossing test is a plain sign check on the simulated `IEQ`; the
within transform lives inside the estimator and never reaches this comparison. Firms already
distressed in 2018 are excluded; the remainder are the prediction set.

This is where Part 1's coordinate fix pays off. With the shipped `IEQ_z`, "the path went negative"
means "the path fell below the panel average", and the predicted crossing rate is meaningless.

In [ ]:
if RUN_E5:
    banner("Part 7 - E5 first-passage validation")
    terms5 = library_terms(E1_BEST)
    t_fit = T_BURN                       # observations 0..T_BURN-1 = 2013-2018
    Xt, Xn, nf = make_pairs(X, None, 0, t_fit - 1)
    fit5 = panel_stlsq(theta(Xt, terms5), Xn, nf, E1_THR, E1_RIDGE)
    resid = Xn - (fit5["alpha"].repeat(t_fit - 1, axis=0) + theta(Xt, terms5) @ fit5["Xi"])
    print(f"fitted on {YEARS[0]}-{YEARS[t_fit - 1]} ({fit5['n_terms']} terms); "
          f"residual rows {resid.shape[0]}, residual sd (mean over states) "
          f"{resid.std(axis=0).mean():.3f}")

    at_risk = np.flatnonzero(Y_DIST[:, t_fit - 1] == 0)
    realised = (Y_DIST[at_risk, t_fit:].max(axis=1) > 0).astype(float)
    steps = T - t_fit
    print(f"at risk at {YEARS[t_fit - 1]}: {len(at_risk)} firms; "
          f"realised entries {YEARS[t_fit]}-{YEARS[-1]}: {int(realised.sum())}")

    rng = np.random.default_rng(SEED + 5)
    n_at = len(at_risk)
    hits = np.zeros(n_at)
    x0 = X[at_risk, t_fit - 1, :]
    a0 = fit5["alpha"][at_risk]
    ieq0_neg = x0[:, I_IEQ] < 0
    block = max(1, N_PATHS_E5 // 20)
    done = 0
    while done < N_PATHS_E5:
        b = min(block, N_PATHS_E5 - done)
        x = np.repeat(x0, b, axis=0)
        a = np.repeat(a0, b, axis=0)
        prev_neg = np.repeat(ieq0_neg, b)
        entered = np.zeros(n_at * b, bool)
        for _ in range(steps):
            e = resid[rng.integers(0, resid.shape[0], size=x.shape[0])]   # whole rows
            x = np.clip(a + theta(x, terms5) @ fit5["Xi"] + e, -CLIP, CLIP)
            neg = x[:, I_IEQ] < 0
            entered |= (neg & prev_neg)
            prev_neg = neg
        hits += entered.reshape(n_at, b).sum(axis=1)
        done += b
    p_model = hits / N_PATHS_E5

    def auc(score, y):
        """Rank-based AUC; score is 'higher means more likely to enter'."""
        order = np.argsort(score, kind="mergesort")
        r = np.empty(len(score), float)
        r[order] = np.arange(1, len(score) + 1)
        # average ranks over ties
        s = np.asarray(score)[order]
        i = 0
        while i < len(s):
            j = i
            while j + 1 < len(s) and s[j + 1] == s[i]:
                j += 1
            if j > i:
                r[order[i:j + 1]] = np.mean(r[order[i:j + 1]])
            i = j + 1
        n1, n0 = y.sum(), (1 - y).sum()
        if n1 == 0 or n0 == 0:
            return float("nan")
        return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

    p_triv = -x0[:, I_IEQ]                       # rank by current IEQ, ascending
    brier_model = float(np.mean((p_model - realised) ** 2))
    base_rate = float(realised.mean())
    brier_base = float(np.mean((base_rate - realised) ** 2))
    auc_model, auc_triv = auc(p_model, realised), auc(p_triv, realised)

    print(f"\n{'':22}{'AUC':>8}{'Brier':>10}")
    print(f"{'dynamical model':22}{auc_model:8.3f}{brier_model:10.4f}")
    print(f"{'rank by current IEQ':22}{auc_triv:8.3f}{'':>10}")
    print(f"{'base rate':22}{'':>8}{brier_base:10.4f}")
    print(f"\npredicted mean P(entry within {steps} yr) = {p_model.mean():.4f}; "
          f"realised {base_rate:.4f}  "
          f"(over-prediction factor {p_model.mean() / max(base_rate, 1e-9):.1f}x)")

    # Discrimination and calibration are separate claims and can disagree. Say which is which.
    disc = ("beats" if auc_model > auc_triv else "does not beat")
    cal = ("is better calibrated than" if brier_model < brier_base
           else "is WORSE calibrated than")
    verdict5 = (f"E5: on discrimination the dynamical model {disc} ranking by current IEQ "
                f"(AUC {auc_model:.3f} vs {auc_triv:.3f}); on calibration it {cal} "
                f"predicting the base rate for everyone "
                f"(Brier {brier_model:.4f} vs {brier_base:.4f}). "
                f"Both belong in the paper - design §5 E5 asks for the reliability diagram "
                f"precisely because a model can rank well and still be badly scaled.")
    print(verdict5)

    RESULTS["E5"] = dict(library=E1_BEST, n_at_risk=int(n_at),
                         n_realised=int(realised.sum()), n_paths=N_PATHS_E5,
                         auc_model=auc_model, auc_trivial=auc_triv,
                         brier_model=brier_model, brier_base_rate=brier_base,
                         mean_predicted=float(p_model.mean()), base_rate=base_rate,
                         verdict=verdict5)

    fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.6))
    edges = np.quantile(p_model, np.linspace(0, 1, 6))
    edges = np.unique(edges)
    if len(edges) > 2:
        b = np.clip(np.digitize(p_model, edges[1:-1]), 0, len(edges) - 2)
        xs = [p_model[b == k].mean() for k in range(len(edges) - 1) if (b == k).any()]
        ys = [realised[b == k].mean() for k in range(len(edges) - 1) if (b == k).any()]
        ax[0].plot([0, max(max(xs), max(ys), 1e-3)], [0, max(max(xs), max(ys), 1e-3)],
                   "k:", lw=0.8)
        ax[0].plot(xs, ys, "o-")
    ax[0].set_xlabel("predicted P(entry)"); ax[0].set_ylabel("observed frequency")
    ax[0].set_title("E5 reliability"); ax[0].grid(alpha=0.3)
    ax[1].hist(p_model, bins=30, color="C0", alpha=0.85)
    ax[1].axvline(base_rate, color="r", ls="--", lw=1,
                  label=f"realised rate {base_rate:.3f}")
    ax[1].set_xlabel(f"predicted P(entry within {steps} yr)"); ax[1].set_ylabel("firms")
    ax[1].set_title("Predicted first-passage probabilities")
    ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
    plt.tight_layout()
    show_fig("E5_first_passage")
else:
    print("[skip] RUN_E5 = False")

---

# Part 8 — Summary

Everything above, collected into one JSON so the write-up quotes numbers this run produced rather
than numbers someone remembers. `RESULTS` also carries `notebook_version`, so a stale figure can
always be traced back to the run that made it.

In [ ]:
banner("Summary")
print(f"version   : {NOTEBOOK_VERSION}")
print(f"quick     : {QUICK}")
print(f"wall clock: {time.time() - _T_START:.1f} s")
print()
if RUN_SYNTH:
    print(f"gate      : synthetic recovery "
          f"{'PASS' if RESULTS['synthetic']['passed'] else 'FAIL'} "
          f"(F1 {RESULTS['synthetic']['best_f1']:.3f})")
if RUN_E1:
    print(f"E1        : nested horizon-{H_SELECT} RMSE "
          f"{RESULTS['E1']['mean_rmse_h3']:.4f}, selected {E1_BEST} @ {E1_THR:.4f}; "
          f"gate {'PASSED' if RESULTS['E1']['gate_passed'] else 'NOT passed'}")
if RUN_E2:
    tbl = [r for r in RESULTS["E2"]["table"] if "oracle" not in r["model"]]
    w = [r for r in tbl if r["beats_ar1"]]
    orc = next((r for r in RESULTS["E2"]["table"] if "oracle" in r["model"]), None)
    print(f"E2        : {len(w)} of {len(tbl)} models beat AR(1)+FE with a CI excluding zero"
          + (": " + ", ".join(r["model"] for r in w) if w else ""))
    if orc:
        print(f"            oracle alpha (not a competitor) sits at "
              f"{orc['rmse_h3']:.4f}, i.e. {-orc['d_vs_ar1']:.4f} of the gap to AR(1)+FE is "
              f"firm-effect estimation noise, not the map")
if RUN_E3:
    print(f"E3        : p(T) = {RESULTS['E3']['p_T']:.4f}, "
          f"{RESULTS['E3']['n_terms_fdr10']} terms at FDR < 0.10")
if RUN_E5:
    print(f"E5        : AUC {RESULTS['E5']['auc_model']:.3f} vs "
          f"{RESULTS['E5']['auc_trivial']:.3f} for the trivial rule; "
          f"Brier {RESULTS['E5']['brier_model']:.4f} vs "
          f"{RESULTS['E5']['brier_base_rate']:.4f} for the base rate")

out = OUT_DIR / "results_financialdist.json"
out.write_text(json.dumps(RESULTS, indent=2, default=float), encoding="utf-8")
print(f"\n[json] {out}")
print(f"[figs] {FIG_DIR}")